# Modelos Generativos con Hugging Face

En esta sección se exploran modelos generativos de lenguaje utilizando la librería Hugging Face.

A diferencia de los enfoques anteriores, estos modelos no solo extraen información, sino que son capaces de generar texto en lenguaje natural.

Se trabajará sobre el corpus del libro *Cien años de soledad* previamente cargado y procesado.

## Objetivos

- Generar texto a partir de un contexto
- Realizar resumen automático
- Construir respuestas en lenguaje natural
- Comparar con modelos de Question Answering

In [7]:
!pip install transformers torch PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.4 MB/s eta 0:00:00


In [8]:
import PyPDF2
from transformers import pipeline

In [ ]:
# Instalación (solo si es necesario)
!pip install transformers torch

In [18]:
text = ""

with open("100as.pdf", "rb") as file:
    reader = PyPDF2.PdfReader(file)

    for page in reader.pages:
        text += page.extract_text()

print("Texto cargado")
print(text[:1500])

Texto cargado
          
Gabriel García Márquez 
Cien años de soledad 
Para Jomi García Ascot 
y María Luisa Elio Cien años de soledad 
Gabriel  García Márquez 
 3  
I 
 
Muchos años después, frente al pelotón de fusilamiento, el coronel Aureliano Buendía había de 
recordar aquella tarde remota en que su padre lo llevó a conocer el hielo. Macondo era entonces 
una aldea de veinte casas de barro y cañabrava construidas a la orilla de un río de aguas diáfanas que se precipitaban por un lecho de piedras pulidas, blancas y enormes como huevos prehistóricos. El mundo era tan reciente, que muchas cosas carecían de nombre, y para 
mencionarlas había que señalarías con el dedo. Todos los años, por el mes de marzo, una familia 
de gitanos desarrapados plantaba su carpa cerca de la aldea, y con un grande alboroto de pitos y timbales daban a conocer los nuevos inventos. Primero llevaron el imán. Un gitano corpulento, de 
barba montaraz y manos de gorrión, que se presentó con el nombre de Melquiad

In [19]:
fragmento = text[:2000]

In [20]:
from transformers import pipeline

In [21]:
# Load and process the document
!pip install -qU  langchain-community pypdf langchain-text-splitters
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("/content/100as.pdf")
texts = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, # Define appropriate chunk size
    chunk_overlap   = 100, # Define appropriate chunk overlap
    length_function = len,
    is_separator_regex = False,
)
docs = text_splitter.split_documents(texts)

print(f"Loaded {len(docs)} document chunks.")

Loaded 1024 document chunks.


## 1. Resumen automático

Se utiliza un modelo generativo para resumir un fragmento del texto.

In [22]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

texto = docs[0].page_content # Ensure we pass string content, not Document object

inputs = tokenizer([texto], max_length=1024, return_tensors="pt", truncation=True)

# Generate Summary
summary_ids = model.generate(
    inputs["input_ids"],
    num_beams=4,
    max_length=80,
    min_length=30,
    early_stopping=True
)

resumen = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Resumen:")
print(resumen)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Resumen:
Gabriel García Márquez Cien años de soledad. Gabriel Garcia Mámquez  Cien a years of soledAD.


### Análisis

El modelo genera un resumen del texto original.

- No copia el contenido
- Reduce la información a lo esencial
- Produce una nueva representación del texto

Esto lo diferencia de modelos como QA, que solo extraen fragmentos.

## 2. Generación de texto explicativo

Se utiliza un modelo instruccional que genera respuestas completas.

In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

prompt = "Explicá quién es Aureliano Buendía en el libro Cien años de soledad:"

inputs = tokenizer(prompt, return_tensors="pt")

respuesta_ids = model.generate(inputs["input_ids"], max_new_tokens=100)

respuesta = tokenizer.decode(respuesta_ids[0], skip_special_tokens=True)

print("Respuesta generada:")
print(respuesta)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Respuesta generada:
Explicá quién es Aureliano Buenda en el libro Cien aos de soledad:


### Análisis

El modelo interpreta la consigna y genera una respuesta completa.

- No depende de un fragmento específico
- Integra conocimiento previo
- Produce lenguaje natural coherente

## 3. Generación con contexto

Se combina un fragmento del texto con una pregunta.

In [26]:
context_text = docs[0].page_content

prompt = f"""
Contexto: {context_text}

Pregunta: ¿Quién llevó a Aureliano a conocer el hielo?

Respuesta:
"""

inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)

respuesta_ids = model.generate(inputs["input_ids"], max_new_tokens=100, num_beams=4, early_stopping=True)

respuesta_con_contexto = tokenizer.decode(respuesta_ids[0], skip_special_tokens=True)

print("Respuesta con contexto:")
print(respuesta_con_contexto)

Respuesta con contexto:
Márquez


### Análisis

El modelo combina:

- comprensión del contexto
- generación de texto

Esto permite respuestas más completas que los modelos de QA tradicionales.

## 4. Comparación con Question Answering

Pregunta: ¿Quién es Aureliano Buendía?

- QA → devuelve fragmentos del texto (ej: "el coronel")
- Generativo → construye una explicación completa

Esto muestra una diferencia clave en el enfoque de los modelos.

## Reflexión

Los modelos generativos permiten una interacción más natural con el lenguaje.

Ventajas:
- Generan respuestas completas
- Son más flexibles
- Permiten múltiples tareas

Limitaciones:
- Pueden generar información incorrecta
- No garantizan precisión absoluta
- Requieren análisis crítico

## Conclusión

Los modelos generativos representan el estado actual del NLP.

A diferencia de los modelos anteriores, no buscan respuestas en el texto, sino que las construyen.

Esto permite resolver tareas más complejas, aunque introduce desafíos en términos de confiabilidad.